# Модуль 1: Chunking стратегии

## Зачем нужен chunking?

RAG (Retrieval-Augmented Generation) работает так:
1. **Индексация**: документы разбиваются на чанки → каждый чанк превращается в embedding (вектор)
2. **Поиск**: вопрос пользователя → embedding → поиск ближайших чанков
3. **Генерация**: найденные чанки + вопрос → LLM → ответ

**Проблема:** если чанки слишком большие — embedding "размывается", поиск неточный.
Если слишком маленькие — теряется контекст, ответ неполный.

**Chunking — это самый важный этап RAG-пайплайна.** Плохой chunking → плохой поиск → плохой ответ.

В этом ноутбуке:
- 4 стратегии chunking (от простой к продвинутой)
- Сравнение на реальном тексте
- Best practices для production

In [1]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    TokenTextSplitter,
)
import tiktoken

print("Всё импортировано!")

Всё импортировано!


## Тестовый документ

Юридический текст (близко к LegalBPM) — хорошо показывает разницу между стратегиями.

In [2]:
LEGAL_DOC = """
# Регламент обработки обращений граждан

## 1. Общие положения

Настоящий регламент устанавливает порядок обработки обращений граждан в электронной форме. Регламент распространяется на все подразделения организации, участвующие в процессе обработки обращений.

Обращение гражданина — это направленное в организацию письменное предложение, заявление или жалоба, а также устное обращение гражданина. Каждое обращение должно быть зарегистрировано в системе в течение одного рабочего дня с момента поступления.

## 2. Классификация обращений

### 2.1 По типу

Обращения классифицируются следующим образом:
- **Предложение** — рекомендация по совершенствованию деятельности организации
- **Заявление** — просьба о содействии в реализации прав и свобод
- **Жалоба** — требование о восстановлении нарушенных прав

### 2.2 По приоритету

Приоритет обращения определяется автоматически на основе следующих критериев:
- **Высокий**: жалобы на нарушение прав, обращения от льготных категорий граждан, повторные обращения
- **Средний**: заявления, требующие межведомственного взаимодействия
- **Низкий**: предложения, информационные запросы

## 3. Процедура обработки

### 3.1 Регистрация

При поступлении обращения оператор выполняет следующие действия:
1. Проверяет полноту заполнения обязательных полей (ФИО, контактные данные, суть обращения)
2. Присваивает регистрационный номер в формате ОГ-ГГГГ-НННННН
3. Определяет тип и приоритет обращения
4. Направляет обращение ответственному исполнителю

### 3.2 Рассмотрение

Ответственный исполнитель обязан:
- Изучить суть обращения в течение 3 рабочих дней
- Запросить дополнительные документы при необходимости
- Подготовить проект ответа
- Согласовать ответ с руководителем подразделения

Срок рассмотрения обращения не должен превышать 30 календарных дней с момента регистрации. В исключительных случаях срок может быть продлён ещё на 30 дней с уведомлением заявителя.

### 3.3 Контроль сроков

Система автоматически отслеживает сроки обработки обращений:
- За 5 дней до истечения срока — уведомление исполнителю
- За 1 день — уведомление исполнителю и руководителю
- При просрочке — эскалация на уровень руководства организации

## 4. Ответственность

За нарушение сроков рассмотрения обращений граждан предусмотрена дисциплинарная ответственность в соответствии с трудовым законодательством Российской Федерации. Повторное нарушение сроков может являться основанием для применения мер дисциплинарного взыскания.
"""

enc = tiktoken.get_encoding("cl100k_base")
print(f"Длина документа: {len(LEGAL_DOC)} символов")
print(f"Токенов (cl100k): {len(enc.encode(LEGAL_DOC))}")

Длина документа: 2456 символов
Токенов (cl100k): 1044


---

## Стратегия 1: Character Splitting (наивный)

Самый простой подход — режем текст по фиксированному количеству символов.

**Плюсы:** просто, предсказуемый размер чанков
**Минусы:** режет посреди слов и предложений, ломает смысл

In [3]:
char_splitter = CharacterTextSplitter(
    separator="",        # без разделителя — режем по символам
    chunk_size=300,       # 300 символов на чанк
    chunk_overlap=50,     # 50 символов перекрытие
)

char_chunks = char_splitter.split_text(LEGAL_DOC)

print(f"Количество чанков: {len(char_chunks)}")
for i, chunk in enumerate(char_chunks[:3]):
    print(f"\n--- Чанк {i+1} ({len(chunk)} символов) ---")
    print(chunk)

Количество чанков: 10

--- Чанк 1 (299 символов) ---
# Регламент обработки обращений граждан

## 1. Общие положения

Настоящий регламент устанавливает порядок обработки обращений граждан в электронной форме. Регламент распространяется на все подразделения организации, участвующие в процессе обработки обращений.

Обращение гражданина — это направленно

--- Чанк 2 (300 символов) ---
обращений.

Обращение гражданина — это направленное в организацию письменное предложение, заявление или жалоба, а также устное обращение гражданина. Каждое обращение должно быть зарегистрировано в системе в течение одного рабочего дня с момента поступления.

## 2. Классификация обращений

### 2.1 По

--- Чанк 3 (300 символов) ---
пления.

## 2. Классификация обращений

### 2.1 По типу

Обращения классифицируются следующим образом:
- **Предложение** — рекомендация по совершенствованию деятельности организации
- **Заявление** — просьба о содействии в реализации прав и свобод
- **Жалоба** — требование о восстано

**Обрати внимание:** чанки разрезаны посреди предложений. Это плохо для RAG — embedding не может нормально "понять" оборванный текст.

---

## Стратегия 2: Recursive Character Splitting (рекомендуемый)

Пытается разрезать по естественным границам: сначала по `\n\n`, потом по `\n`, потом по `. `, потом по ` `.

**Это дефолтный и самый популярный сплиттер в LangChain.**

**Плюсы:** сохраняет предложения и абзацы целыми
**Минусы:** размер чанков менее предсказуемый

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

recursive_chunks = recursive_splitter.split_text(LEGAL_DOC)

print(f"Количество чанков: {len(recursive_chunks)}")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\n--- Чанк {i+1} ({len(chunk)} символов) ---")
    print(chunk)

---

## Стратегия 3: Token-based Splitting

Режет по количеству токенов, а не символов. Это важно потому что:
- LLM имеют лимит контекста в **токенах**, не в символах
- Embedding модели тоже принимают фиксированное число токенов (512, 1024, 8192)
- 1 русский символ ≈ 2-3 токена (в отличие от английского, где ~1 символ ≈ 1 токен)

**Плюсы:** точный контроль размера для LLM/embedding
**Минусы:** медленнее (нужна токенизация)

In [ ]:
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # кодировка GPT-4 / text-embedding-3
    chunk_size=150,               # 150 токенов на чанк
    chunk_overlap=20,             # 20 токенов перекрытие
)

token_chunks = token_splitter.split_text(LEGAL_DOC)
enc = tiktoken.get_encoding("cl100k_base")

print(f"Количество чанков: {len(token_chunks)}")
for i, chunk in enumerate(token_chunks[:3]):
    tokens = len(enc.encode(chunk))
    print(f"\n--- Чанк {i+1} ({len(chunk)} символов, {tokens} токенов) ---")
    print(chunk)

---

## Стратегия 4: Markdown Header Splitting (document-aware)

Самый умный подход для структурированных документов — **разбиваем по заголовкам**.
Каждый чанк получает **метаданные**: к какому разделу он относится.

**Идеально для юридических документов** (регламенты, инструкции, НПА).

**Плюсы:** сохраняет структуру документа, добавляет metadata
**Минусы:** работает только со структурированными документами

In [ ]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Документ"),
        ("##", "Раздел"),
        ("###", "Подраздел"),
    ],
    strip_headers=False,
)

md_chunks = md_splitter.split_text(LEGAL_DOC)

print(f"Количество чанков: {len(md_chunks)}")
for i, chunk in enumerate(md_chunks):
    print(f"\n--- Чанк {i+1} ---")
    print(f"Metadata: {chunk.metadata}")
    print(f"Текст ({len(chunk.page_content)} символов):")
    print(chunk.page_content[:200])
    if len(chunk.page_content) > 200:
        print("...")

**Обрати внимание на metadata!** Каждый чанк знает, к какому разделу документа он относится. Можно использовать для:
- Фильтрации при поиске ("ищи только в разделе 3")
- Добавления контекста в prompt ("Из раздела 'Процедура обработки':")
- Аналитики (какие разделы чаще запрашивают)

---

## Сравнение стратегий

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

def stats(name, chunks):
    texts = [c if isinstance(c, str) else c.page_content for c in chunks]
    sizes = [len(t) for t in texts]
    token_sizes = [len(enc.encode(t)) for t in texts]
    print(f"\n{name}:")
    print(f"  Чанков: {len(chunks)}")
    print(f"  Символов: min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)//len(sizes)}")
    print(f"  Токенов:  min={min(token_sizes)}, max={max(token_sizes)}, avg={sum(token_sizes)//len(token_sizes)}")

stats("Character (наивный)", char_chunks)
stats("Recursive (рекомендуемый)", recursive_chunks)
stats("Token-based", token_chunks)
stats("Markdown Headers", md_chunks)

---

## Комбинированный подход (production best practice)

В production часто комбинируют:
1. **Сначала** Markdown Header Splitting — разбиваем по структуре
2. **Потом** Recursive Splitting — дробим большие секции

Так получаем и структурные метаданные, и контролируемый размер чанков.

In [ ]:
# Шаг 1: Разбиваем по заголовкам
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Документ"),
        ("##", "Раздел"),
        ("###", "Подраздел"),
    ],
    strip_headers=False,
)
header_chunks = md_splitter.split_text(LEGAL_DOC)

# Шаг 2: Большие секции дробим recursive-сплиттером
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

final_chunks = []
for chunk in header_chunks:
    if len(chunk.page_content) > 400:
        sub_chunks = recursive_splitter.create_documents(
            [chunk.page_content],
            metadatas=[chunk.metadata],
        )
        final_chunks.extend(sub_chunks)
    else:
        final_chunks.append(chunk)

print(f"После комбинированного подхода: {len(final_chunks)} чанков\n")
for i, chunk in enumerate(final_chunks):
    print(f"Чанк {i+1} | {len(chunk.page_content)} символов | {chunk.metadata}")
    print(f"  {chunk.page_content[:100]}...\n")

---

## Параметры chunking: chunk_size и chunk_overlap

### chunk_size (размер чанка)

| Размер | Когда использовать |
|--------|-------------------|
| 100-200 токенов | Точный поиск по фактам (FAQ, определения) |
| 200-500 токенов | Баланс (большинство случаев) |
| 500-1000 токенов | Контекстный поиск (нарративы, юридические тексты) |

### chunk_overlap (перекрытие)

- Обычно 10-20% от chunk_size
- Нужно чтобы не терять контекст на границе чанков
- Слишком большой overlap → дублирование, шум при поиске

### Для русского языка

Русский текст занимает ~2-3x больше токенов чем английский.
Если embedding модель принимает 512 токенов — это ~200-250 символов русского текста (vs ~400-500 англ.).

**Правило:** для русского текста ставь chunk_size в токенах (TokenTextSplitter), а не в символах.

In [ ]:
# Эксперимент: как chunk_size влияет на результат (комбинированный подход)

for size in [200, 400, 600]:
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Документ"),
            ("##", "Раздел"),
            ("###", "Подраздел"),
        ],
        strip_headers=False,
    )
    header_chunks = md_splitter.split_text(LEGAL_DOC)

    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=size // 8,  # overlap ~12.5% от chunk_size
    )

    chunks = []
    for chunk in header_chunks:
        if len(chunk.page_content) > size:
            sub = recursive_splitter.create_documents(
                [chunk.page_content], metadatas=[chunk.metadata]
            )
            chunks.extend(sub)
        else:
            chunks.append(chunk)

    sizes = [len(c.page_content) for c in chunks]
    print(f"chunk_size={size} | overlap={size // 8}")
    print(f"  Чанков: {len(chunks)} | Размеры: {min(sizes)}-{max(sizes)} символов (avg {sum(sizes)//len(sizes)})")
    for i, c in enumerate(chunks):
        print(f"  [{i+1}] {len(c.page_content):>3} симв. | {c.metadata}")
    print()

---

## Практическое задание

Попробуй самостоятельно:
1. Измени `chunk_size` на 200 и 600 — как меняется количество и качество чанков?
2. Убери `chunk_overlap` (поставь 0) — что теряется на границах?
3. Добавь свой текст (можно из LegalBPM knowledge base) и примени комбинированный подход

## Выводы

| Стратегия | Когда использовать |
|-----------|-------------------|
| Character | Никогда (только для понимания проблемы) |
| **Recursive** | **Дефолт для большинства случаев** |
| Token-based | Когда важен точный контроль размера в токенах |
| **Markdown Headers** | **Структурированные документы (юридические, технические)** |
| **Комбинированный** | **Production — headers + recursive** |

**Следующий модуль:** Embeddings и векторные базы данных — как превратить чанки в векторы и искать по ним.